In [1]:
import os
import re
from tqdm import tqdm
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict, Counter
import random
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
size_file = "large"

In [4]:
df_behaviors = pd.read_csv(f"/content/drive/MyDrive/MIND{size_file}_train/behaviors.tsv", sep="\t", names=['ImpressionID', 'UserID', 'Time', 'History', 'Impressions'])
df_news = pd.read_csv(f"/content/drive/MyDrive/MIND{size_file}_train/news.tsv", sep="\t", names=['NewsID', 'Category', 'SubCategory', 'Title', 'Abstract', 'URL', 'TitleEntities', 'AbstractEntities'])

In [5]:
df_behaviors_val = pd.read_csv(f"/content/drive/MyDrive/MIND{size_file}_dev/behaviors.tsv", sep="\t", names=['ImpressionID', 'UserID', 'Time', 'History', 'Impressions'])
df_news_val = pd.read_csv(f"/content/drive/MyDrive/MIND{size_file}_dev/news.tsv", sep="\t", names=['NewsID', 'Category', 'SubCategory', 'Title', 'Abstract', 'URL', 'TitleEntities', 'AbstractEntities'])

In [6]:
df_behaviors.shape

(2232748, 5)

In [7]:
df_behaviors["Time"] = pd.to_datetime(df_behaviors["Time"])
cutoff = pd.to_datetime("2019-11-14")

behavior_train = df_behaviors[df_behaviors["Time"] < cutoff].copy()
behavior_val   = df_behaviors[df_behaviors["Time"] >= cutoff].copy()

In [8]:
df_news.shape

(101527, 8)

In [9]:
df_news = pd.concat([df_news, df_news_val])
df_news = df_news.drop_duplicates(subset='NewsID')

In [10]:
df_news.shape

(110842, 8)

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

Usando dispositivo: cuda


In [12]:
def tokenize(text):
    tokens = re.findall(r"[\w']+", text.lower())
    return tokens

In [13]:
longitudes = df_news["Title"].dropna().apply(lambda x: len(x.split()))
cantidad_menor_20 = (longitudes < 20).sum()
total = len(longitudes)

print(f"Títulos con menos de 20 palabras: {cantidad_menor_20} de {total} ({cantidad_menor_20 / total:.2%})")

Títulos con menos de 20 palabras: 109507 de 110842 (98.80%)


In [14]:
word2idx = {'<PAD>': 0, '<UNK>': 1}
idx = 2 # Por <UNK> y <PAD>
news2idx = {}  # Mapeo: news_id -> lista de índices de palabras (padded/trunc)
max_size_title = 20

In [15]:
for _, row in tqdm(df_news.iterrows(), total=df_news.shape[0]):
    news_id = row["NewsID"]
    title = row["Title"]
    tokens = [] if pd.isna(title) else tokenize(title)
    token_idxs = []
    for w in tokens[:max_size_title]:  # truncar título largo
        if w not in word2idx:
            word2idx[w] = idx
            idx += 1
        token_idxs.append(word2idx.get(w, word2idx['<UNK>']))
    # Rellenar con PAD si es más corto que title_max
    if len(token_idxs) < max_size_title:
        token_idxs += [word2idx['<PAD>']] * (max_size_title - len(token_idxs))
    news2idx[news_id] = token_idxs

100%|██████████| 110842/110842 [00:06<00:00, 17330.47it/s]


In [16]:
vocab_size = len(word2idx)
print(f'Vocabulario: {vocab_size} palabras')

Vocabulario: 50684 palabras


In [17]:
data = []
for _, row in tqdm(behavior_train.iterrows(), total=behavior_train.shape[0]):
    hist_str = row['History']
    hist_ids = [] if pd.isna(hist_str) else [nid for nid in hist_str.split() if nid]
    impr = row['ImpressionID']
    imps = [] if pd.isna(row['Impressions']) else row['Impressions'].split()
    for imp in imps:
        if len(imp) == 0:
            continue
        parts = imp.split('-')
        if len(parts) != 2:
            continue
        news_id, click = parts[0], parts[1]
        label = int(click)
        data.append((impr, hist_ids, news_id, label))

100%|██████████| 1801231/1801231 [03:26<00:00, 8709.50it/s] 


In [18]:
print(f'Total de ejemplos de interacción: {len(data)}')

Total de ejemplos de interacción: 66107268


In [19]:
val_data = []

for _, row in tqdm(behavior_val.iterrows(), total=behavior_val.shape[0]):
    hist_str = row['History']
    hist_ids = [] if pd.isna(hist_str) else [nid for nid in hist_str.split() if nid]
    impr = row['ImpressionID']
    imps = row['Impressions'].split()
    for imp in imps:
        if len(imp) == 0:
            continue
        parts = imp.split('-')
        if len(parts) != 2:
            continue
        news_id, click = parts[0], parts[1]
        val_data.append((impr, hist_ids, news_id, int(click)))

100%|██████████| 431517/431517 [00:49<00:00, 8791.18it/s] 


In [20]:
print(f'Total ejemplos validación: {len(val_data)}')

Total ejemplos validación: 17400106


In [21]:
class MINDListDataset(Dataset):
    """
    Devuelve:
        hist_tensor  : [hist_max, title_max]
        cand_tensor  : [C,         title_max]
        label_tensor : [C]  (0/1, un solo 1)
        impr_id      : str
    """
    def __init__(self, interactions, news2idx, word2idx,
                 hist_max, title_max):
        self.news2idx  = news2idx
        self.word2idx  = word2idx
        self.hist_max  = hist_max
        self.title_max = title_max

        # Agrupar ejemplos por impresión -----------------------------
        sessions = defaultdict(list)
        for impr, hist_ids, cand_id, label in interactions:
            sessions[impr].append((hist_ids, cand_id, label))
        self.impr_ids = list(sessions.keys())
        self.sessions = sessions

    def __len__(self):
        return len(self.impr_ids)

    def __getitem__(self, idx):
        impr   = self.impr_ids[idx]
        triples = self.sessions[impr]           # lista de (hist, cand, label)

        # -------- historial (todos los candidatos comparten el mismo) -----
        hist_ids = triples[0][0][-self.hist_max:]        # recorte por la derecha
        hist_seq = [self.news2idx.get(nid,
                    [self.word2idx['<PAD>']]*self.title_max) for nid in hist_ids]
        while len(hist_seq) < self.hist_max:
            hist_seq.insert(0, [self.word2idx['<PAD>']]*self.title_max)

        # -------- candidatos + etiquetas -------------------------------
        cand_seqs, labels = [], []
        for _, cand_id, lbl in triples:
            cand_seqs.append(
                self.news2idx.get(cand_id,
                     [self.word2idx['<PAD>']]*self.title_max))
            labels.append(lbl)

        return (torch.tensor(hist_seq,  dtype=torch.long),        # [H,L]
                torch.tensor(cand_seqs, dtype=torch.long),        # [C,L]
                torch.tensor(labels,   dtype=torch.float),        # [C]
                impr)

In [22]:
def collate_fn_list(batch):
    """
    Devuelve:
        hist_batch  : [B, H, L]
        cand_batch  : [B, C_max, L]
        label_batch : [B, C_max]  (0/1, padded con -1)
        mask_batch  : [B, C_max]  (True donde existe candidato)
        impr_batch  : list[str]
    """
    hist_list, cand_list, label_list, impr_list = zip(*batch)

    # Historial: tamaño fijo
    hist_batch = torch.stack(hist_list)             # [B,H,L]

    # Candidatos: pad al máximo C del batch
    C_max = max(x.size(0) for x in cand_list)
    L     = cand_list[0].size(1)
    pad_val = 0  # token PAD

    cand_pad   = torch.full((len(batch), C_max, L), pad_val, dtype=torch.long)
    label_pad  = torch.full((len(batch), C_max),    -1,      dtype=torch.float)
    mask_pad   = torch.zeros(len(batch), C_max,     dtype=torch.bool)

    for i,(cands, labels) in enumerate(zip(cand_list, label_list)):
        C = cands.size(0)
        cand_pad[i,:C]  = cands
        label_pad[i,:C] = labels
        mask_pad[i,:C]  = 1

    return (hist_batch.to(device),
            cand_pad.to(device),
            label_pad.to(device),
            mask_pad.to(device),
            list(impr_list))


In [23]:
max_hist_title = 50
batch_size = 32

In [24]:
train_dataset = MINDListDataset(data, news2idx, word2idx, max_hist_title, max_size_title)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn_list)

val_dataset = MINDListDataset(val_data, news2idx, word2idx, max_hist_title, max_size_title)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_list)

In [25]:
embed_dim = 300
num_heads = 20
lr = 0.001

In [26]:
glove = True

if glove:
    embedding_matrix = np.random.normal(scale=0.6, size=(vocab_size, embed_dim))
    found = 0
    with open("/content/drive/MyDrive/MINDlarge_dev/glove.6B.300d.txt", 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split(' ')
            word = parts[0]
            if word in word2idx:
                vec = np.array(parts[1:], dtype=np.float32)
                if vec.shape[0] == embed_dim:
                    embedding_matrix[word2idx[word]] = vec
                    found += 1
    print(f'Palabras encontradas en GloVe: {found}/{vocab_size}')
    embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float)
else:
    embedding_matrix = None

Palabras encontradas en GloVe: 36351/50684


In [27]:
class FastformerAttention(nn.Module):
    """
    Atención Fastformer (Atención aditiva global) que reemplaza nn.MultiheadAttention.
    Opera con entradas de forma (L, B, E) o (B, L, E), realizando la proyección Q, K, V
    por separado, obteniendo vectores globales y propagando interacciones por producto
    elemento a elemento, según Fastformer (Fastformer: Additive Attention Can Be All You Need).
    """
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super(FastformerAttention, self).__init__()
        assert embed_dim % num_heads == 0, "embed_dim debe ser divisible por num_heads"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        # Proyecciones lineales para Q, K, V (similar a multi-cabeza estándar)
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=True)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=True)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=True)
        # Parámetros de atención aditiva por cabeza (vectores de peso para Q y K)
        # Formato (num_heads, head_dim) para aplicar dot-product con cada vector de dimensión head_dim
        self.attn_wq = nn.Parameter(torch.Tensor(num_heads, self.head_dim))
        self.attn_wk = nn.Parameter(torch.Tensor(num_heads, self.head_dim))
        # Capa de salida tras concatenar cabezas
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=True)
        self.dropout = nn.Dropout(dropout)
        # Inicialización
        nn.init.xavier_uniform_(self.W_q.weight)
        nn.init.xavier_uniform_(self.W_k.weight)
        nn.init.xavier_uniform_(self.W_v.weight)
        nn.init.xavier_uniform_(self.out_proj.weight)
        nn.init.zeros_(self.W_q.bias)
        nn.init.zeros_(self.W_k.bias)
        nn.init.zeros_(self.W_v.bias)
        nn.init.zeros_(self.out_proj.bias)
        nn.init.xavier_uniform_(self.attn_wq)
        nn.init.xavier_uniform_(self.attn_wk)

    def forward(self, query, key, value):
        """
        query, key, value: tensores de forma (L, B, E) ó (S, N, E) donde
        L=longitud de secuencia, B=batch, E=embed_dim.
        Fastformer es simétrico en q=k=v, pero aceptamos tres argumentos para compatibilidad.
        """
        # Permutar para batch-first: [B, L, E]
        transpose = False
        if query.dim() == 3 and query.shape[0] != query.shape[1]:
            # Asumimos forma (L, B, E)
            query = query.transpose(0, 1)
            key = key.transpose(0, 1)
            value = value.transpose(0, 1)
            transpose = True
        # Proyectar Q, K, V
        # Ahora shapes: [B, L, E]
        Q = self.W_q(query)   # [B, L, E]
        K = self.W_k(key)     # [B, L, E]
        V = self.W_v(value)   # [B, L, E]
        B, L, E = Q.size()
        H = self.num_heads
        D = self.head_dim
        # Dividir en cabezas: [B, L, H, D]
        Q = Q.view(B, L, H, D)
        K = K.view(B, L, H, D)
        V = V.view(B, L, H, D)
        # Reordenar para [B, H, L, D]
        Q = Q.permute(0, 2, 1, 3)
        K = K.permute(0, 2, 1, 3)
        V = V.permute(0, 2, 1, 3)
        # =========== Fastformer Steps ===========
        # 1) Atención aditiva sobre Q para obtener q_global [B, H, D]
        # Calculamos puntuaciones: sum_{j}( w_q[h,j] * Q[...,j] )
        # w_q: [H, D], Q: [B, H, L, D]
        # Producto elemento a elemento y sumar sobre dimensión D:
        # scores_q: [B, H, L]
        scores_q = (Q * self.attn_wq.unsqueeze(0).unsqueeze(2)).sum(dim=-1)  # [B, H, L]
        alpha = torch.softmax(scores_q, dim=-1)  # [B, H, L]
        # Obtener vector q_global: suma ponderada de Q sobre L
        # alpha: [B, H, L], Q: [B, H, L, D] -> q_global: [B, H, D]
        q_global = torch.einsum('bhl,bhld->bhd', alpha, Q)
        # 2) Interactuar q_global con cada K por producto elemento a elemento -> K'
        # Extendemos q_global para cada posición L: [B, H, 1, D] * [B, H, L, D] -> [B, H, L, D]
        K_prime = q_global.unsqueeze(2) * K  # [B, H, L, D]
        # 3) Atención aditiva sobre K_prime para obtener k_global [B, H, D]
        scores_k = (K_prime * self.attn_wk.unsqueeze(0).unsqueeze(2)).sum(dim=-1)  # [B, H, L]
        beta = torch.softmax(scores_k, dim=-1)  # [B, H, L]
        k_global = torch.einsum('bhl,bhld->bhd', beta, K_prime)  # [B, H, D]
        # 4) Interactuar k_global con cada V -> V'
        V_prime = k_global.unsqueeze(2) * V  # [B, H, L, D]
        # Rearmar V' combinando cabezas: [B, H, L, D] -> [B, L, H*D]
        V_prime = V_prime.permute(0, 2, 1, 3).contiguous().view(B, L, H * D)  # [B, L, E]
        # Capa lineal de salida y agregar Q (residuo)
        out = self.out_proj(V_prime)  # [B, L, E]
        # Capa residual: sumamos la proyección original de Q antes de dividir cabezas
        # Primero reconstruir Q original (bidimensional por cada posición)
        Q_orig = Q.permute(0, 2, 1, 3).contiguous().view(B, L, H * D)  # [B, L, E]
        out = out + Q_orig
        # Opcional: aplicar dropout
        out = self.dropout(out)
        # Devolver en forma (L, B, E)
        if transpose:
            out = out.transpose(0, 1).contiguous()
        return out

In [28]:
class NewsEncoder(nn.Module):
    """
    Codificador de noticias: procesa títulos de noticias (secuencias de tokens)
    y produce vectores de noticia. Reemplaza la atención multi-cabeza por FastformerAttention.
    """
    def __init__(self, vocab_size, embed_dim, num_heads, title_max, pretrained_emb=None):
        super(NewsEncoder, self).__init__()
        self.embed_dim = embed_dim
        self.title_max = title_max
        # Capa de embedding de palabras
        self.word_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        if pretrained_emb is not None:
            self.word_embedding.weight.data.copy_(pretrained_emb)
            self.word_embedding.weight.requires_grad = True  # o False si no quieres fine-tune

        # Capa convolucional 1D para extraer características locales de palabras (opcional, similar a arquitectura original)
        # Usamos múltiples filtros 1xD para captar n-gramas de tamaño 3 por ejemplo
        self.conv = nn.Conv1d(in_channels=embed_dim, out_channels=embed_dim, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        # Atención Fastformer sobre la secuencia de características de palabras
        self.self_attn = FastformerAttention(embed_dim, num_heads, dropout=0.1)
        # Atención aditiva para agregar las palabras importantes en el título
        self.attn_vector = nn.Linear(embed_dim, 1)  # para puntuación de cada palabra
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        """
        x: tensor de tokens de noticias con forma [B, title_max].
        Devuelve: vectores de noticias de forma [B, embed_dim].
        """
        # Embedding y extracción de características locales
        # Palabras: [B, title_max, E] -> conv espera [B, E, title_max]
        emb = self.word_embedding(x)            # [B, L, E]
        emb = emb.transpose(1, 2)               # [B, E, L]
        conv_out = self.relu(self.conv(emb))    # [B, E, L]
        conv_out = conv_out.transpose(1, 2)     # [B, L, E]
        # Atención Fastformer (auto-atención) entre las posiciones de palabras
        # FastformerAttention espera (L, B, E) o (B, L, E); adaptamos:
        conv_out_trans = conv_out.transpose(0, 1).contiguous()  # [L, B, E]
        attn_out = self.self_attn(conv_out_trans, conv_out_trans, conv_out_trans)  # [L, B, E]
        attn_out = attn_out.transpose(0, 1)  # [B, L, E]
        # Atención aditiva para obtener vector final de noticia
        # Calcular puntuación de importancia para cada palabra
        scores = self.attn_vector(attn_out).squeeze(-1)  # [B, L]
        weights = self.softmax(scores)                    # [B, L]
        news_vector = torch.bmm(weights.unsqueeze(1), attn_out).squeeze(1)  # [B, E]
        return news_vector  # [B, embed_dim]

class UserEncoder(nn.Module):
    """
    Codificador de usuario: agrega vectores de noticias historiales usando Fastformer.
    Toma un historial de noticias y devuelve un vector de usuario.
    """
    def __init__(self, news_encoder, embed_dim, num_heads, hist_max):
        super(UserEncoder, self).__init__()
        self.news_encoder = news_encoder  # instancia de NewsEncoder para codificar cada noticia
        self.hist_max = hist_max
        # Atención Fastformer sobre la secuencia de vectores de noticia del historial
        self.self_attn = FastformerAttention(embed_dim, num_heads, dropout=0.1)
        # Atención aditiva para agregar las noticias más relevantes del historial
        self.attn_vector = nn.Linear(embed_dim, 1)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, hist_x):
        """
        hist_x: tensor de tokens de noticias de historial con forma [B, hist_max, title_max].
        Devuelve: vector de usuario de forma [B, embed_dim].
        """
        B, H, L = hist_x.size()
        # Codificar cada noticia en el historial
        hist_x_flat = hist_x.view(B * H, L)                  # [B*H, title_max]
        news_vectors = self.news_encoder(hist_x_flat)        # [B*H, embed_dim]
        news_vectors = news_vectors.view(B, H, -1)           # [B, hist_max, embed_dim]
        # Atención Fastformer sobre las noticias del historial
        nv_trans = news_vectors.transpose(0, 1).contiguous() # [H, B, E]
        attn_out = self.self_attn(nv_trans, nv_trans, nv_trans)  # [H, B, E]
        attn_out = attn_out.transpose(0, 1)                  # [B, H, E]
        # Atención aditiva para agregar vectores de noticias importantes
        scores = self.attn_vector(attn_out).squeeze(-1)      # [B, H]
        weights = self.softmax(scores)                       # [B, H]
        user_vector = torch.bmm(weights.unsqueeze(1), attn_out).squeeze(1)  # [B, E]
        return user_vector  # [B, embed_dim]

In [29]:
class FastformerNRMS(nn.Module):
    """
    Modelo NRMS modificado con Fastformer.
    Mantiene la misma interfaz: forward(hist_tensor, cand_tensor).
    hist_tensor: [B, hist_max, title_max], cand_tensor: [B, cand_count, title_max].
    """
    def __init__(self, vocab_size, embed_dim, num_heads, title_max, hist_max, pretrained_emb=None):
        super(FastformerNRMS, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.title_max = title_max
        self.hist_max = hist_max
        # News encoder y user encoder con Fastformer
        self.news_encoder = NewsEncoder(vocab_size, embed_dim, num_heads, title_max, pretrained_emb)
        self.user_encoder = UserEncoder(self.news_encoder, embed_dim, num_heads, hist_max)
        # (Opcional) proyección final o dropout
        self.dropout = nn.Dropout(0.1)

    def forward(self, hist_tensor, cand_tensor, mask=None):
        """
        hist_tensor : [B, hist_max, title_max]
        cand_tensor : [B, C,       title_max]
        mask        : [B, C]  (bool) – True donde el candidato existe
        """
        B, H, L = hist_tensor.size()
        _, C, _ = cand_tensor.size()
        # Codificar usuario
        user_vector = self.user_encoder(hist_tensor)            # [B, E]
        # Codificar candidatos (aplicar NewsEncoder a cada candidato)
        cand_flat = cand_tensor.view(B * C, L)                  # [B*C, title_max]
        cand_vecs = self.news_encoder(cand_flat)               # [B*C, E]
        cand_vecs = cand_vecs.view(B, C, -1)                   # [B, cand_count, E]
        # Calcular similaridad (producto punto usuario con cada candidato)
        # Expandir user_vector para combinar con candidatos

        cand_vecs = self.dropout(cand_vecs)
        user_vector = self.dropout(user_vector)

        user_exp = user_vector.unsqueeze(1)                    # [B, 1, E]
        logits = torch.sum(cand_vecs * user_exp, dim=-1)       # [B, cand_count]

        if mask is not None:
            logits = logits.masked_fill(~mask, -1e9)             # -∞ donde no hay candidato

        return logits                                            # [B,C]

In [30]:
model = FastformerNRMS(vocab_size, embed_dim, num_heads, max_size_title, max_hist_title,
             pretrained_emb=embedding_matrix.to(device) if embedding_matrix is not None else None)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

In [31]:
unique_seqs = list({tuple(seq) for seq in news2idx.values()})  # set → list
seq2idx     = {seq: i for i, seq in enumerate(unique_seqs)}
print(f"Noticias únicas: {len(unique_seqs):,}")

# ------------------------------------------------------------------
# 2) Popularidad (clics positivos)
#    data : lista de tuplas (impr_id, hist_ids, news_id, label)
# ------------------------------------------------------------------
from collections import Counter
import numpy as np

pop_cnt = Counter()
for _, _, nid, label in data:          # behaviour de entrenamiento
    if label == 1:                     # cuenta SOLO clics
        seq = tuple(news2idx[nid])
        pop_cnt[seq] += 1

news_popularity = np.zeros(len(unique_seqs), dtype=np.int32)
for seq, c in pop_cnt.items():
    news_popularity[seq2idx[seq]] = c

Noticias únicas: 98,743


In [32]:
def build_news_vectors(model, unique_seqs, batch_size=2048, device="cuda"):
    """
    Devuelve un np.array [N, embed_dim] con los vectores de cada noticia,
    procesando en lotes para no reventar la GPU.
    """
    model.eval()
    vecs = []

    with torch.no_grad():
        for i in range(0, len(unique_seqs), batch_size):
            batch = torch.tensor(
                unique_seqs[i:i + batch_size],
                dtype=torch.long,
                device=device
            )                               # [B, title_len]

            v = model.news_encoder(batch)   # [B, E]    (solo fwd)
            vecs.append(v.cpu())            # mueve a CPU y libera GPU

            del batch, v
            torch.cuda.empty_cache()

    return torch.cat(vecs, dim=0).numpy()   # [N, E] en NumPy

In [33]:
all_vectors = build_news_vectors(model, unique_seqs, batch_size=2048)

In [34]:
def ndcg_score(labels, scores, k=5):
    order = np.argsort(scores)[::-1]
    labels = np.array(labels)
    dcg = 0.0
    for i in range(min(k, len(labels))):
        rel = labels[order[i]]
        dcg += (2**rel - 1) / np.log2(i+2)
    ideal = np.sort(labels)[::-1]
    idcg = 0.0
    for i in range(min(k, int(np.sum(labels)))):
        idcg += 1.0 / np.log2(i+2)
    return dcg / idcg if idcg > 0 else 0.0

def mrr_score(labels, scores):
    order = np.argsort(scores)[::-1]
    labels = np.array(labels)[order]
    for rank, label in enumerate(labels, start=1):
        if label == 1:
            return 1.0 / rank
    return 0.0

def novelty_score(idx_list, popularity_arr):
    """Mayor es mejor (novedad = -log popularidad)."""
    pops = popularity_arr[idx_list] + 1e-8        # evitar log(0)
    probs = pops / pops.sum()
    return float(np.mean(-np.log2(probs)))

def diversity_score(idx_list, vector_bank):
    """1 − similitud media coseno (mayor = más diverso)."""
    if len(idx_list) <= 1:
        return 0.0
    vecs = vector_bank[idx_list]                  # [k, E]
    sims = cosine_similarity(vecs)
    upper = sims[np.triu_indices_from(sims, k=1)]
    return float(1.0 - upper.mean())

In [35]:
epochs = 2

In [ ]:
best_auc = 0.0
best_model_state = None

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    for hist_batch, cand_batch, label_batch, mask_batch, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}"):
        optimizer.zero_grad()
        logits = model(hist_batch, cand_batch, mask_batch)
        target = label_batch.argmax(dim=1)        # [B]
        loss = criterion(logits, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch} - Pérdida promedio: {avg_loss:.4f}")

    model.eval()
    ndcg5_list, ndcg10_list, mrr_list, auc_list = [], [], [], []
    novelty_list = []
    diversity_list = []

    with torch.no_grad():
        for hist_batch, cand_batch, label_batch, mask_batch, impr_batch in tqdm(val_loader, total = len(val_loader)):
            logits  = model(hist_batch, cand_batch, mask_batch)  # [B,C]
            scores  = logits.cpu().numpy()
            labels  = label_batch.cpu().numpy()
            masks   = mask_batch.cpu().numpy()
            cands   = cand_batch.cpu().numpy()

            for s, l, m, cseq in zip(scores, labels, masks, cands):
                # recortar a candidatos reales
                s = s[m]        # (C_real,)
                l = l[m]        # (C_real,)

                cseq = cseq[m]

                ndcg5_list .append(ndcg_score(l, s, k=5))
                ndcg10_list.append(ndcg_score(l, s, k=10))
                mrr_list  .append(mrr_score(l, s))
                if l.max() > 0 and l.min() == 0:     # al menos 1 pos y 1 neg
                    auc_list.append(roc_auc_score(l, s))

                topk = np.argsort(s)[::-1][:10]
                seq_keys  = [tuple(row) for row in cseq[topk]]        # cada noticia = tuple(tokens)
                idx_list = [seq2idx.get(seq, -1) for seq in seq_keys]  # -1 si es “nueva”
                idx_list = [i for i in idx_list if i >= 0]             # quita las nuevas
                if not idx_list:                                       # todos eran nuevos
                    continue

                novelty_list  .append(novelty_score(idx_list, news_popularity))
                diversity_list.append(diversity_score(idx_list, all_vectors))

    ndcg5  = np.mean(ndcg5_list)
    ndcg10 = np.mean(ndcg10_list)
    mrr    = np.mean(mrr_list)
    auc    = np.mean(auc_list) if auc_list else 0.0
    novelty   = np.mean(novelty_list)
    diversity = np.mean(diversity_list)

    if auc > best_auc:
        best_auc = auc
        best_model_state = model.state_dict()
        print(f"» Nuevo mejor modelo guardado (AUC = {auc:.4f})")

    print(f"Validación – AUC: {auc:.4f} | MRR: {mrr:.4f} | "
      f"nDCG@5: {ndcg5:.4f} | nDCG@10: {ndcg10:.4f} | "
      f"Novedad: {novelty:.4f} | Diversidad: {diversity:.4f}")

if best_model_state is not None:
    torch.save(best_model_state, "nrms_fastformer_best.pt")
    print("Modelo con mejor AUC guardado en nrms_fastformer_best.pt")

Epoch 1/2: 100%|██████████| 56289/56289 [52:44<00:00, 17.79it/s]


Epoch 1 - Pérdida promedio: 48140762.2011


100%|██████████| 13485/13485 [21:43<00:00, 10.35it/s]


» Nuevo mejor modelo guardado (AUC = 0.6552)
Validación – AUC: 0.6552 | MRR: 0.3371 | nDCG@5: 0.3120 | nDCG@10: 0.3756 | Novedad: 14.9607 | Diversidad: 0.0845


Epoch 2/2: 100%|██████████| 56289/56289 [52:30<00:00, 17.86it/s]


Epoch 2 - Pérdida promedio: 9606427.2138


100%|██████████| 13485/13485 [21:32<00:00, 10.43it/s]


» Nuevo mejor modelo guardado (AUC = 0.6689)
Validación – AUC: 0.6689 | MRR: 0.3549 | nDCG@5: 0.3271 | nDCG@10: 0.3903 | Novedad: 15.0008 | Diversidad: 0.0867
Modelo con mejor AUC guardado en nrms_fastformer_best.pt


In [ ]:
import os, shutil

src_path  = "nrms_fastformer_best.pt"                   # archivo que ya tienes
dst_dir   = "/content/drive/MyDrive/MINDlarge_train"    # carpeta en Drive
shutil.copy(src_path, dst_dir)

'/content/drive/MyDrive/MINDlarge_train/nrms_fastformer_best.pt'

In [36]:
model_path = "/content/drive/MyDrive/MINDlarge_train/nrms_fastformer_best.pt"
model.load_state_dict(torch.load(model_path, map_location=device))

<All keys matched successfully>

In [37]:
def construir_seq2title(news2idx, idx2word, pad_idx):
    seq2title = {}
    for token_ids in news2idx.values():
        seq_key = tuple(token_ids)                         # <-- clave exactamente igual a lo que entrega el DataLoader
        words = [idx2word[i] for i in token_ids if i != pad_idx]
        seq2title[seq_key] = " ".join(words)
    return seq2title

In [38]:
def mostrar_perfil_y_recomendaciones(model, dataloader, seq2title, device, n=3, hist_k=5, stop_on_click=True):
    model.eval()
    ejemplos_mostrados = 0

    with torch.no_grad():
        for hist_batch, cand_batch, label_batch, mask_batch, _ in dataloader:
            # Mover a device
            hist_batch = hist_batch.to(device)
            cand_batch = cand_batch.to(device)
            label_batch = label_batch.to(device)
            mask_batch = mask_batch.to(device)

            logits = model(hist_batch, cand_batch, mask_batch)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            labels = label_batch.cpu().numpy()
            masks = mask_batch.cpu().numpy()

            indices = list(range(len(hist_batch)))
            random.shuffle(indices)  # Mezcla los usuarios dentro del batch

            for i in indices:
                print(f"\n--- Usuario {ejemplos_mostrados+1} ---")

                # Historial (máx hist_k)
                print(f"Historial (hasta {hist_k} ítems):")
                contador = 0
                for seq in hist_batch[i].cpu().numpy():
                    titulo = seq2title.get(tuple(seq), None)
                    if titulo:
                        print(f" - {titulo}")
                        contador += 1
                        if contador >= hist_k:
                            break

                # Candidatos
                print("\nCandidatos (ordenados por score):")
                scores = probs[i]
                cand_seqs = cand_batch[i].cpu().numpy()
                mask_line = masks[i].astype(bool)
                valid_idx = scores[mask_line].argsort()[::-1]
                real_pos = np.where(mask_line)[0][valid_idx]

                for j in real_pos:
                    seq = tuple(cand_seqs[j])
                    titulo = seq2title.get(seq, "[Título desconocido]")
                    marcador = "✔" if labels[i][j] == 1 else "✖"
                    print(f"  ({marcador}) {titulo}")
                    if stop_on_click and labels[i][j] == 1:
                        break

                ejemplos_mostrados += 1
                if ejemplos_mostrados >= n:
                    return

In [39]:
PAD = word2idx['<PAD>']
idx2word = {v: k for k, v in word2idx.items()}
seq2title = construir_seq2title(news2idx, idx2word, PAD)

# Ejemplos de recomendaciones a usuarios

"n" indica la cantidad de usuarios de ejemplo
"hist_k" la cantidad de noticias del historial que se mostrará
"stop_on_click" si se quieren mostrar todos los candidatos o solo hasta que el usuario hizo click. Las noticias candidatas están ordenadas por score

In [43]:
print("\n\nMostrando ejemplos de perfiles y recomendaciones del modelo:")
mostrar_perfil_y_recomendaciones(model, val_loader, seq2title, device, n=3, hist_k=3, stop_on_click=True)



Mostrando ejemplos de perfiles y recomendaciones del modelo:

--- Usuario 1 ---
Historial (hasta 3 ítems):
 - the best celebrity couple halloween costumes
 - ricky martin and husband jwan yosef welcome their fourth child a baby boy
 - property brothers' j d scott marries annalee belle in vintage theatre themed wedding

Candidatos (ordenados por score):
  (✖) kelly osbourne on mom sharon and chrissy teigen's tiff 'it's okay if people disagree'
  (✖) marcia cross' anal cancer may have been linked to hpv she wants people to know they could have the virus
  (✖) miranda lambert is pretty in pink at 2019 cma awards with husband brendan mcloughlin
  (✖) olympic swimmer ryan lochte went from earning millions to living paycheck to paycheck here's what he learned
  (✖) 9 amazing transgender women who changed history
  (✖) the latest on brad pitt and angelina jolie's post split relationship plus more news
  (✖) gospel singer tamela mann lost 40 pounds by overhauling her lifestyle
  (✔) watch ke

In [ ]:
print("\n\nMostrando ejemplos de perfiles y recomendaciones del modelo:")
mostrar_perfil_y_recomendaciones(model, val_loader, seq2title, device, n=5, hist_k=5, stop_on_click=True)



Mostrando ejemplos de perfiles y recomendaciones del modelo:

--- Usuario 1 ---
Historial (hasta 5 ítems):
 - an architect who built his dream tiny home in colorado shares photos
 - woman spots deadly animal hiding in photo of her kids
 - michelle pfeiffer initially blamed herself after metoo moment 'i should've known'
 - can you spot the cat hiding among the bats in this spooky halloween brainteaser
 - he grew a 910 pound pumpkin and then used it as a boat

Candidatos (ordenados por score):
  (✔) this is chick fil a's most ordered menu item

--- Usuario 2 ---
Historial (hasta 5 ítems):
 - snakehead fish that survives on land was discovered in georgia officials want it dead
 - couple didn't know why car was running strangely then they popped the hood
 - farmers in idaho rallied to harvest a neighbor's potatoes as a deep freeze threatened to ruin them
 - hennessey venom f5's engine makes 1 817 hp fury comes standard
 - how elizabeth warren could 'vaporize' america's oil boom

Candidat

In [71]:
print("\n\nMostrando ejemplos de perfiles y recomendaciones del modelo:")
mostrar_perfil_y_recomendaciones(model, val_loader, seq2title, device, n=1, hist_k=1000, stop_on_click=True)



Mostrando ejemplos de perfiles y recomendaciones del modelo:

--- Usuario 1 ---
Historial (hasta 1000 ítems):
 - robert forster oscar nommed star of 'jackie brown ' dies at 78
 - eliud kipchoge runs 1 59 marathon first to break 2 hours
 - patriots reportedly trying to acquire a star wide receiver
 - evacuation orders lifted in saddleridge fire zone as firefighters make progress
 - dmx heading back to rehab cancels concerts
 - report ezekiel elliott's dad investigated after police shoot wild cat near his home
 - former nfl lineman justin bannan arrested for attempted murder
 - contractor claims video shows structural flaws prior to hard rock hotel collapse
 - suspect nicknamed 'woo woo' charged in three murders claims to have committed five more 'i want to be a serial
 - mark hurd oracle ceo who led 3 tech companies dies at 62
 - gabbard hits back at 'queen of warmongers' clinton
 - alcs game 5 yankees stay alive with win against astros
 - lapd detectives identify third victim of sexu

In [82]:
print("\n\nMostrando ejemplos de perfiles y recomendaciones del modelo:")
mostrar_perfil_y_recomendaciones(model, val_loader, seq2title, device, n=1, hist_k=1000, stop_on_click=True)



Mostrando ejemplos de perfiles y recomendaciones del modelo:

--- Usuario 1 ---
Historial (hasta 1000 ítems):
 - the news in cartoons
 - zimbabwe sent 30 baby elephants to china says rights group
 - stephen king recommends the movie tv show and book you should check out this halloween
 - flynn hearing canceled after lawyer claims fbi manipulated files
 - 'tarzan' actor's son unarmed when fatally shot by deputies
 - opinions 10 reasons the democrats are winning on impeachment
 - the news in cartoons
 - why president trump's kids are probably celebrating his move to florida
 - heidi klum's 2019 halloween costume transformation is mind blowing but like what is it
 - pamela anderson gets backlash after wearing a native american headdress for halloween
 - hugh hefner's son cooper weds 'harry potter' actress
 - us rep ilhan omar divorces husband in minnesota
 - anthony mackie and wife quietly divorced last year report
 - trailer dolittle
 - nike hit with another damning op ed 'i was emotio

In [86]:
print("\n\nMostrando ejemplos de perfiles y recomendaciones del modelo:")
mostrar_perfil_y_recomendaciones(model, val_loader, seq2title, device, n=1, hist_k=1000, stop_on_click=True)



Mostrando ejemplos de perfiles y recomendaciones del modelo:

--- Usuario 1 ---
Historial (hasta 1000 ítems):
 - fortnite's black hole has closed and chapter 2 is finally here
 - nfl world reacts to officials handing packers win over lions
 - harley davidson halts production of new electric motorcycles
 - charlize theron margot robbie have some face time in l a plus priyanka chopra jonas will smith more
 - woman suspect dead at 'tarzan' actor ron ely's california residence
 - this man's rattlesnake bite is a warning to everyone to take animal bites more seriously
 - here's why we pass out candy on halloween
 - fort worth shooting officers weren't asked to do welfare check here's how it changed things
 - hawk can't understand why this little 'bunny' isn't scared of him
 - burt reynolds' former 1978 'smokey' pontiac trans am in big auction by feds
 - emily ratajkowski is being sued for 150 000 over an instagram photo
 - i gave up on love and it was one of the best decisions i ever made

In [87]:
print("\n\nMostrando ejemplos de perfiles y recomendaciones del modelo:")
mostrar_perfil_y_recomendaciones(model, val_loader, seq2title, device, n=1, hist_k=1000, stop_on_click=True)



Mostrando ejemplos de perfiles y recomendaciones del modelo:

--- Usuario 1 ---
Historial (hasta 1000 ítems):
 - snakehead fish that survives on land was discovered in georgia officials want it dead
 - couple didn't know why car was running strangely then they popped the hood
 - farmers in idaho rallied to harvest a neighbor's potatoes as a deep freeze threatened to ruin them
 - hennessey venom f5's engine makes 1 817 hp fury comes standard
 - how elizabeth warren could 'vaporize' america's oil boom
 - 'a passionate guy' ex nascar team owner found dead in ohio river near louisville
 - this is saleen's new gt4 race car
 - dick's sporting goods ceo quietly tests presidential bid
 - eric tse 24 just became a billionaire overnight
 - the permian basin is facing its biggest threat yet
 - jeff bezos lost about 7 billion on thursday
 - florida needs python hunters a man in iran is one of thousands applying for the job
 - 'go back to work' outcry over deaths on amazon's warehouse floor
 - na

In [89]:
print("\n\nMostrando ejemplos de perfiles y recomendaciones del modelo:")
mostrar_perfil_y_recomendaciones(model, val_loader, seq2title, device, n=1, hist_k=1000, stop_on_click=True)



Mostrando ejemplos de perfiles y recomendaciones del modelo:

--- Usuario 1 ---
Historial (hasta 1000 ítems):
 - a texas mom is going to prison after putting her son through unnecessary medical procedures
 - ohio voters express angst over impeachment
 - amazon is shipping expired baby formula and beef jerky putting big brands at risk
 - tornado touched down in northern dallas national weather service says
 - 3 row suvs compared explorer telluride palisade enclave and cx 9
 - highest paid musicians in 2019
 - north texas couple moving into new home sees it destroyed by tornado
 - jennifer lawrence hired a food truck for her wedding and the owner had no idea who she was
 - four flight attendants were arrested in miami's airport after bringing in thousands in cash police say
 - why aren't more women getting mammograms
 - chicago mayor set to unveil budget plan for huge deficit
 - 10 best cities to visit in winter
 - why 1 million children were kicked off medicaid
 - how 17 under 50 wome